##### Support Request Agent Stream

Stream support requests through the support agent endpoint and persist reports.

In [ ]:
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().getOrElse(None)
DATABRICKS_HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().getOrElse(None)

CATALOG = dbutils.widgets.get("CATALOG")
SUPPORT_AGENT_ENDPOINT_NAME = dbutils.widgets.get("SUPPORT_AGENT_ENDPOINT_NAME")

In [ ]:
%pip install openai

In [ ]:
import json
from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from openai import OpenAI

BATCH_SIZE = 20


def process_support_request(payload_json: str) -> str:
    payload = json.loads(payload_json)
    support_request_id = payload.get("support_request_id", "")
    user_id = payload.get("user_id", "")
    order_id = payload.get("order_id", "")

    client = OpenAI(
        api_key=DATABRICKS_TOKEN,
        base_url=f"{DATABRICKS_HOST}/serving-endpoints",
    )

    fallback = json.dumps({
        "support_request_id": support_request_id,
        "user_id": user_id,
        "order_id": order_id,
        "credit_recommendation": {"amount_usd": 3.0, "reason": "Fallback goodwill credit."},
        "refund_recommendation": {"amount_usd": 0.0, "reason": "No verified severe failure in fallback mode."},
        "draft_response": "Thanks for contacting us. I reviewed your case and added a $3.00 goodwill credit while we complete a final verification.",
        "past_interactions_summary": "Fallback summary: history unavailable in this attempt.",
        "order_details_summary": "Fallback summary: order details unavailable in this attempt.",
        "decision_confidence": "medium",
        "escalation_flag": False,
    })

    message = "\n".join([
        "You are the Caspers support triage agent.",
        "Use the structured case payload below.",
        "Return strictly valid JSON only with keys:",
        "support_request_id, user_id, order_id, credit_recommendation, refund_recommendation, draft_response, past_interactions_summary, order_details_summary, decision_confidence, escalation_flag.",
        "Do not return placeholders like 'we will look into this'.",
        f"Case payload: {json.dumps(payload)}",
    ])

    def deterministic_final_report() -> str:
        features = payload.get("feature_context") or {}
        request_text = str(payload.get("request_text") or "")
        repeat = int(features.get("repeat_complaints_30d") or 0)
        risk = float(features.get("risk_score") or 0.25)
        policy_limit = float(features.get("policy_limit_usd") or 20.0)

        if repeat >= 2 or risk >= 0.75:
            refund = round(min(policy_limit, max(policy_limit * 0.9, 15.0)), 2)
            credit = round(min(10.0, max(5.0, policy_limit * 0.25)), 2)
            confidence = "high"
        elif repeat == 1 or risk >= 0.45:
            refund = round(min(policy_limit, max(policy_limit * 0.5, 6.0)), 2)
            credit = round(min(7.0, max(3.0, policy_limit * 0.15)), 2)
            confidence = "high"
        else:
            refund = 0.0
            credit = round(min(5.0, max(2.0, policy_limit * 0.1)), 2)
            confidence = "medium"

        if repeat >= 2:
            intro = "I can see this has happened more than once, and that is not acceptable."
        else:
            intro = "I am sorry this happened, and thank you for reporting it."

        draft = (
            f"{intro} We are issuing a ${refund:.2f} refund and adding ${credit:.2f} in credits. "
            f"Issue noted: {request_text}"
        ).strip()

        return json.dumps({
            "support_request_id": support_request_id,
            "user_id": user_id,
            "order_id": order_id,
            "credit_recommendation": {"amount_usd": credit, "reason": "Policy-based support resolution."},
            "refund_recommendation": {"amount_usd": refund, "reason": "Policy-based support resolution."},
            "draft_response": draft,
            "past_interactions_summary": "Computed from available user support history.",
            "order_details_summary": "Computed from available order context.",
            "decision_confidence": confidence,
            "escalation_flag": False,
        })

    for _ in range(3):
        try:
            response_obj = client.responses.create(
                model=SUPPORT_AGENT_ENDPOINT_NAME,
                input=[{"role": "user", "content": message}],
            )
            response = response_obj.output[-1].content[0].text
            parsed = json.loads(response)
            draft = str(parsed.get("draft_response") or "").lower()
            has_placeholder = "look into this" in draft or "reviewing your request" in draft
            bad_ordinal = " 1th " in draft or " 2th " in draft or " 3th " in draft
            missing_amounts = parsed.get("refund_recommendation") in (None, {}, "") and parsed.get("credit_recommendation") in (None, {}, "")
            if has_placeholder or missing_amounts or bad_ordinal:
                return deterministic_final_report()
            return response
        except Exception:
            continue

    return deterministic_final_report()


process_support_request_udf = udf(process_support_request, StringType())

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.support")
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.support.support_agent_reports (
      support_request_id STRING,
      user_id STRING,
      order_id STRING,
      request_text STRING,
      ts TIMESTAMP,
      agent_response STRING
    )
    """
)

# Backfill-safe schema evolution for existing tables.
try:
    spark.sql(f"ALTER TABLE {CATALOG}.support.support_agent_reports ADD COLUMNS (request_text STRING)")
except Exception:
    pass

spark.sql(
    f"""
    ALTER TABLE {CATALOG}.support.support_agent_reports
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
    """
)

# Backfill raw request text for historical rows where it was not stored yet.
spark.sql(
    f"""
    MERGE INTO {CATALOG}.support.support_agent_reports s
    USING (
      SELECT support_request_id, request_text
      FROM {CATALOG}.support.raw_support_requests
    ) r
    ON s.support_request_id = r.support_request_id
    WHEN MATCHED AND s.request_text IS NULL THEN
      UPDATE SET s.request_text = r.request_text
    """
)

pending = spark.sql(
    f"""
    SELECT
      r.support_request_id,
      r.user_id,
      r.order_id,
      r.ts,
      r.request_type,
      COALESCE(r.edge_case_type, 'standard') AS edge_case_type,
      r.request_text,
      COALESCE(f.repeat_complaints_30d, 0) AS repeat_complaints_30d,
      COALESCE(f.policy_limit_usd, 20.0) AS policy_limit_usd,
      COALESCE(f.risk_score, 0.25) AS risk_score
    FROM {CATALOG}.support.raw_support_requests r
    LEFT JOIN {CATALOG}.support.support_request_features f
      ON r.support_request_id = f.support_request_id
    LEFT ANTI JOIN {CATALOG}.support.support_agent_reports s
      ON r.support_request_id = s.support_request_id
    ORDER BY r.ts DESC
    LIMIT {BATCH_SIZE}
    """
)

order_context = spark.sql(
    f"""
    SELECT
      order_id,
      MAX(CASE WHEN event_type = 'order_created' THEN location END) AS location,
      MAX(CASE WHEN event_type = 'order_created' THEN get_json_object(body, '$.items') END) AS items_json,
      MAX(CASE WHEN event_type = 'order_created' THEN get_json_object(body, '$.customer_addr') END) AS customer_address,
      MIN(CASE WHEN event_type = 'order_created' THEN try_to_timestamp(ts) END) AS order_created_ts,
      MAX(CASE WHEN event_type = 'delivered' THEN try_to_timestamp(ts) END) AS delivered_ts
    FROM {CATALOG}.lakeflow.all_events
    GROUP BY order_id
    """
)

history_rows = spark.sql(
    f"""
    SELECT user_id, support_request_id, ts, request_text, request_type, COALESCE(edge_case_type, 'standard') AS edge_case_type
    FROM {CATALOG}.support.raw_support_requests
    """
)

history_latest = (
    history_rows
    .withColumn("rn", F.row_number().over(Window.partitionBy("user_id").orderBy(F.col("ts").desc())))
    .filter(F.col("rn") <= 5)
    .groupBy("user_id")
    .agg(
        F.to_json(
            F.collect_list(
                F.struct(
                    "support_request_id",
                    "ts",
                    "request_type",
                    "edge_case_type",
                    "request_text",
                )
            )
        ).alias("past_request_context_json")
    )
)

enriched = (
    pending
    .join(order_context, on="order_id", how="left")
    .join(history_latest, on="user_id", how="left")
    .withColumn(
        "payload_json",
        F.to_json(
            F.struct(
                F.col("support_request_id"),
                F.col("user_id"),
                F.col("order_id"),
                F.col("request_type"),
                F.col("edge_case_type"),
                F.col("request_text"),
                F.struct(
                    F.col("repeat_complaints_30d"),
                    F.col("policy_limit_usd"),
                    F.col("risk_score"),
                ).alias("feature_context"),
                F.struct(
                    F.col("location"),
                    F.col("items_json"),
                    F.col("customer_address"),
                    F.col("order_created_ts"),
                    F.col("delivered_ts"),
                ).alias("order_context"),
                F.coalesce(F.col("past_request_context_json"), F.lit("[]")).alias("past_request_context_json"),
            )
        ),
    )
)

processed = (
    enriched
    .withColumn("ts", F.current_timestamp())
    .withColumn("agent_response", process_support_request_udf(F.col("payload_json")))
    .select("support_request_id", "user_id", "order_id", "request_text", "ts", "agent_response")
)

processed.write.mode("append").saveAsTable(f"{CATALOG}.support.support_agent_reports")
processed_count = processed.count()
print(f"Processed {processed_count} support requests in this run")

sample_rows = processed.select("support_request_id", "agent_response").limit(3).collect()
for sample in sample_rows:
    print(f"sample_support_request_id={sample['support_request_id']}")
    print(sample["agent_response"][:600])

sample_preview = sample_rows[0]["agent_response"][:600] if sample_rows else ""
dbutils.notebook.exit(
    json.dumps(
        {
            "processed_count": processed_count,
            "sample_preview": sample_preview,
        }
    )
)